# VisiPruner trên SmolVLM · Bản full

Chạy đủ 23 cấu hình trên 4.204 câu test, sau đó chấm điểm và xuất bốn hình. Bật GPU và Internet. Có thể chạy lại từng cell đo để tiếp tục từ các bộ ảnh đã lưu.

Sau baseline, mỗi cell đo chạy tối đa hai cấu hình cùng lúc trên T4 ×2. Với một GPU, cell tự chạy tuần tự.


In [ ]:
%%writefile /kaggle/working/visipruner-ablation.env
REPOSITORY=https://github.com/huyvanzzz/Log_Metrics.git
SOURCE_BRANCH=smolvlm-attention-nguyet
SOURCE_DIR=/kaggle/working/log-metrics-visipruner
DATA_DIR=/kaggle/working/drivelm-nuscenes-v1-0
export HF_TOKEN_SECRET=''
OUTPUT_DIR=/kaggle/working/visipruner_ablation_outputs
BATCH_SIZE=8
MAX_NEW_TOKENS=128


## Cài đặt

Cập nhật mã nguồn và cài thư viện chạy mô hình.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
if [ -d "$SOURCE_DIR/.git" ]; then
  test "$(git -C "$SOURCE_DIR" branch --show-current)" = "$SOURCE_BRANCH"
  test -z "$(git -C "$SOURCE_DIR" status --porcelain --untracked-files=no)"
  git -C "$SOURCE_DIR" pull --ff-only origin "$SOURCE_BRANCH"
else
  git clone --depth 1 --single-branch --branch "$SOURCE_BRANCH" "$REPOSITORY" "$SOURCE_DIR"
fi
python -m pip install -q -r "$SOURCE_DIR/smolvlm_attention/requirements.txt" pycocoevalcap==1.2
if ! command -v java >/dev/null; then
  apt-get update -qq
  apt-get install -y -qq default-jre-headless
fi


## Chuẩn bị dữ liệu

Tải split test đã ghim, lấy mẫu phân tầng và ghi các vị trí merge vào manifest.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.mvpruner_ablation fetch-hf --output "$DATA_DIR"
python -u -m smolvlm_attention.visipruner_ablation prepare --test "$DATA_DIR/multi_frame/multi_frame_test.json" --image-root "$DATA_DIR" --output "$OUTPUT_DIR"


## 01/23 · Baseline · SmolVLM với đủ token ảnh

Sinh đáp án baseline và lưu thứ hạng token ảnh ở các layer dùng để mask.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run --variant baseline --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 02–03

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 02/23 · Layer 1–2 · Mask top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

### 03/23 · Layer 1–2 · Chỉ giữ top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants shallow-top10 shallow-keep10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 04–05

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 04/23 · Layer 1 · Mask toàn bộ attention lên ảnh

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

### 05/23 · Layer 1 · Merge attention với seed 42

Dồn visual attention vào patch đã chọn theo camera và vùng ảnh. Giữ nguyên tổng trọng số cho mỗi query và head.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants shallow-none shallow-merge-42 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 06–07

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 06/23 · Layer 1 · Merge attention với seed 43

Dồn visual attention vào patch đã chọn theo camera và vùng ảnh. Giữ nguyên tổng trọng số cho mỗi query và head.

### 07/23 · Layer 1 · Merge attention với seed 44

Dồn visual attention vào patch đã chọn theo camera và vùng ảnh. Giữ nguyên tổng trọng số cho mỗi query và head.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants shallow-merge-43 shallow-merge-44 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 08–09

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 08/23 · Layer 9–15 · Mask top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

### 09/23 · Layer 9–15 · Mask bottom 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-9-15-top10 middle-9-15-bottom10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 10–11

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 10/23 · Layer 9–15 · Chỉ giữ top 5%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

### 11/23 · Layer 9–15 · Mask top 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-9-15-keep5 middle-9-15-top10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 12–13

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 12/23 · Layer 9–15 · Mask bottom 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

### 13/23 · Layer 18–24 · Mask top 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-9-15-bottom10-mass middle-18-24-top10 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 14–15

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 14/23 · Layer 18–24 · Mask bottom 10%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

### 15/23 · Layer 18–24 · Chỉ giữ top 5%

Can thiệp sau softmax trong prefill. Lưu tổng lượng attention bị lấy đi.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-18-24-bottom10 middle-18-24-keep5 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 16–17

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 16/23 · Layer 18–24 · Mask top 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

### 17/23 · Layer 18–24 · Mask bottom 10% và bảo toàn visual mass

Mask token đã chọn, sau đó phân bổ lại trọng số lên các token ảnh còn lại. Giữ nguyên trọng số lên text.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants middle-18-24-top10-mass middle-18-24-bottom10-mass --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 18–19

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 18/23 · Layer 20 · Skip token ảnh

Bỏ qua xử lý visual tokens tại đúng layer này. Các visual tokens quay lại ở layer kế tiếp.

### 19/23 · Layer 20 · Discard token ảnh

Loại visual tokens khỏi chuỗi và KV cache từ layer này đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-skip-20 deep-discard-20 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 20–21

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 20/23 · Layer 23 · Skip token ảnh

Bỏ qua xử lý visual tokens tại đúng layer này. Các visual tokens quay lại ở layer kế tiếp.

### 21/23 · Layer 23 · Discard token ảnh

Loại visual tokens khỏi chuỗi và KV cache từ layer này đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-skip-23 deep-discard-23 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Lượt 22–23

Hai cấu hình dùng hai GPU khi có sẵn. Với một GPU, chúng chạy lần lượt.

### 22/23 · Layer 26 · Skip token ảnh

Bỏ qua xử lý visual tokens tại đúng layer này. Các visual tokens quay lại ở layer kế tiếp.

### 23/23 · Layer 26 · Discard token ảnh

Loại visual tokens khỏi chuỗi và KV cache từ layer này đến hết.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation run-pair --variants deep-skip-26 deep-discard-26 --output "$OUTPUT_DIR" --image-root "$DATA_DIR" --batch-size "$BATCH_SIZE" --max-new-tokens "$MAX_NEW_TOKENS"


## Chấm điểm và vẽ hình

Xuất bảng điểm và bốn hình PNG/PDF. Bước này chạy trên CPU.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation score --output "$OUTPUT_DIR"


In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/visipruner-ablation.env
cd "$SOURCE_DIR"
python -u -m smolvlm_attention.visipruner_ablation plot --output "$OUTPUT_DIR"
